# 02 — DGCNN (Dynamical Graph Convolutional Neural Network)

**Family:** Graph CNN &nbsp;•&nbsp; **Input:** DE features `[14 x 5]`

DGCNN (Song et al., 2020, *IEEE TAC*) treats the 14 electrodes as nodes of a
graph whose **adjacency matrix is a trainable parameter**, learned jointly with
the classifier rather than fixed from electrode distance. Graph filtering uses a
K-order Chebyshev polynomial of the normalized graph Laplacian.

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.data_loader import load_subject
from src.models import dgcnn

set_seed(42)
DEVICE = get_device()
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
LOSO_SPLITS = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR = os.path.join(ROOT, "results")

# ----- experiment configuration -----
MODEL_NAME = "DGCNN"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/dgcnn.py`: two `ChebyGraphConv` layers (order **K = 2**, see the
`# ASSUMPTION` note in the source) over a learnable `[14 x 14]` adjacency, ReLU +
dropout between them, then a 2-layer MLP head. Node features are the 5 band DE
values per electrode.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = dgcnn.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## Validation protocol — subject-independent (LOSO)

The benchmark uses a single protocol so every architecture is scored identically:
**23-fold leave-one-subject-out** (`data/splits/loso_splits.json`). Fold *k* trains a
fresh model on 22 subjects and tests on the held-out 23rd; normalisation statistics
are fit on the training subjects only and a 10% slice of them is used for early
stopping. We report the **mean ± std of the headline metrics across the 23 folds**.

Everything runs through `src.benchmark_si.run_one` — the one entry point shared by
every model notebook — which reads this model's recipe from `benchmark_si.REGISTRY`,
trains through `src.evaluation.run_loso`, and writes `results/{MODEL_NAME}_results.json`.


## 3. Run the subject-independent (LOSO) benchmark

In [ ]:
# One shared call: 23-fold LOSO for this model, then writes results/DGCNN_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## 4. Learned electrode graph (interpretability)

DGCNN's learned adjacency is a thesis figure in its own right: it shows which
electrode pairs the model found informative for emotion, independent of physical
distance. Below we train one quick model on LOSO fold 0 (22 subjects) and plot its
adjacency.

In [ ]:
import matplotlib.pyplot as plt

from src.data_loader import build_loso_loaders
from src.evaluation import make_default_fit
from src.models.dgcnn import DGCNN

tl, vl, el, meta = build_loso_loaders(
    PROCESSED_DIR, LOSO_SPLITS, fold=0,
    label=LABEL, feature="de", batch_size=256, normalize=True, val_frac=0.0,
)
m = DGCNN(n_nodes=14, in_dim=5, n_classes=2).to(DEVICE)
_ = make_default_fit(epochs=(5 if QUICK else 40), lr=1e-3)(m, tl, None, DEVICE)

A = m.learned_adjacency().numpy()
chans = load_subject(PROCESSED_DIR, meta["test_subject"]).channel_names
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap="viridis")
ax.set_xticks(range(14)); ax.set_xticklabels(chans, rotation=90)
ax.set_yticks(range(14)); ax.set_yticklabels(chans)
ax.set_title("DGCNN learned adjacency (LOSO fold 0)")
fig.colorbar(im); plt.tight_layout()
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, "figures", "dgcnn_adjacency_loso_fold0.png"), dpi=150)
plt.show()

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The per-model LOSO recipe (feature view, batch size, epochs, lr) lives in
  `src/benchmark_si.py::REGISTRY` — a single source of truth shared by this
  notebook and the `python -m src.benchmark_si` CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test the whole flow (3 folds / few epochs); `LABEL`
  switches the target between valence / arousal / dominance with no other change.
